# Phase 11 Ablations on Google Colab GPU

**Instructions:**
1. In the file browser on the left, click the **Upload** icon and upload the `colab_data.zip` file generated on your local machine.
2. Ensure your Colab runtime is set to GPU (Runtime > Change runtime type > Hardware accelerator > T4 GPU).
3. Run all cells below.

In [ ]:
!unzip -o colab_data.zip

In [ ]:
import os
import json
import time
import datetime
import numpy as np
import scipy.sparse as sp
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss, jaccard_score, accuracy_score
from sklearn.multioutput import ClassifierChain
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

TARGET_COLS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

def apply_thresholds(P_matrix, thresholds, label_names):
    Y_pred = np.zeros_like(P_matrix)
    for i, label in enumerate(label_names):
        tau = thresholds[label]
        Y_pred[:, i] = (P_matrix[:, i] >= tau).astype(int)
    return Y_pred

def optimize_thresholds(P_val, Y_val, label_names, tau_range=(0.05, 0.95), step=0.01):
    thresholds = {}
    for i, label in enumerate(label_names):
        best_tau = 0.5
        best_f1 = -1
        for tau in np.arange(tau_range[0], tau_range[1] + step, step):
            preds = (P_val[:, i] >= tau).astype(int)
            score = f1_score(Y_val[:, i], preds, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_tau = tau
        thresholds[label] = float(best_tau)
    return thresholds

def compute_metrics(Y_true, Y_pred, label_names):
    metrics = {
        'macro_f1': float(f1_score(Y_true, Y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(Y_true, Y_pred, average='micro', zero_division=0)),
        'macro_precision': float(precision_score(Y_true, Y_pred, average='macro', zero_division=0)),
        'macro_recall': float(recall_score(Y_true, Y_pred, average='macro', zero_division=0)),
        'hamming_loss': float(hamming_loss(Y_true, Y_pred)),
        'jaccard': float(jaccard_score(Y_true, Y_pred, average='samples', zero_division=0)),
        'subset_accuracy': float(accuracy_score(Y_true, Y_pred)),
        'per_class': {}
    }
    
    for i, label in enumerate(label_names):
        metrics['per_class'][label] = {
            'f1': float(f1_score(Y_true[:, i], Y_pred[:, i], zero_division=0)),
            'precision': float(precision_score(Y_true[:, i], Y_pred[:, i], zero_division=0)),
            'recall': float(recall_score(Y_true[:, i], Y_pred[:, i], zero_division=0))
        }
    return metrics

In [ ]:
print("Loading feature matrices...")
X_train = sp.load_npz("X_train.npz")
X_val   = sp.load_npz("X_val.npz")
X_test  = sp.load_npz("X_test.npz")
Y_train = np.load("Y_train.npy")
Y_val   = np.load("Y_val.npy")
Y_test  = np.load("Y_test.npy")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

# Compute chain orders
freqs = Y_train.sum(axis=0)
freq_desc_indices = np.argsort(freqs)[::-1]
freq_desc_names = [TARGET_COLS[i] for i in freq_desc_indices]
rev_freq_desc_names = freq_desc_names[::-1]
alpha_names = sorted(TARGET_COLS)
rare_middle_names = ["joy", "anger", "fear", "surprise", "disgust", "sadness"]

# Co-occurrence order
coocc = Y_train.T @ Y_train
corr_names = [freq_desc_names[0]]
remaining = list(TARGET_COLS)
remaining.remove(corr_names[0])
while remaining:
    best_next, best_score = None, -1
    for cand in remaining:
        score = sum(coocc[TARGET_COLS.index(cand), TARGET_COLS.index(p)] for p in corr_names)
        if score > best_score:
            best_score = score
            best_next = cand
    corr_names.append(best_next)
    remaining.remove(best_next)


In [ ]:
os.makedirs("ablations_out", exist_ok=True)
RUN_TS = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
all_results = {}

def run_ablation(exp_id, description, C=1.0, class_weight="balanced", chain_order=None, chain_seed=42):
    print(f"\n{'='*70}")
    print(f"Running {exp_id} — {description}")
    print(f"C={C}, CW={class_weight}, Order={chain_order}, Seed={chain_seed}")
    
    t0 = time.time()
    order_indices = [TARGET_COLS.index(name) for name in chain_order]
    
    # 1. Fit (Using GPU doesn't natively speed up sklearn LR, but running in Colab unblocks local resources)
    base_clf = LogisticRegression(C=C, class_weight=class_weight, max_iter=2000, solver="saga", random_state=42)
    chain = ClassifierChain(base_clf, order=order_indices, random_state=chain_seed)
    chain.fit(X_train, Y_train)
    fit_time = time.time() - t0
    print(f"  Fit time: {fit_time:.1f}s")
    
    # 2. Val threshold
    t1 = time.time()
    P_val = chain.predict_proba(X_val)
    thresholds = optimize_thresholds(P_val, Y_val, label_names=TARGET_COLS)
    Y_val_pred = apply_thresholds(P_val, thresholds, TARGET_COLS)
    val_metrics = compute_metrics(Y_val, Y_val_pred, TARGET_COLS)
    print(f"  Val Macro-F1: {val_metrics['macro_f1']:.4f}")
    
    # 3. Test evaluate
    t2 = time.time()
    P_test = chain.predict_proba(X_test)
    Y_test_pred = apply_thresholds(P_test, thresholds, TARGET_COLS)
    test_metrics = compute_metrics(Y_test, Y_test_pred, TARGET_COLS)
    print(f"  Test Macro-F1: {test_metrics['macro_f1']:.4f}")
    
    result = {
        "experiment_id": exp_id,
        "description": description,
        "hyperparameters": {"C": C, "class_weight": class_weight, "chain_order": chain_order, "chain_seed": chain_seed},
        "thresholds": thresholds,
        "validation_metrics": val_metrics,
        "test_metrics": test_metrics,
        "fit_time_seconds": round(fit_time, 2)
    }
    
    with open(f"ablations_out/{exp_id}.json", "w") as f:
        json.dump(result, f, indent=2)
        
    all_results[exp_id] = result
    return result


In [ ]:
# ==========================================
# Group A: LR Regularization (C sweep)
# ==========================================
for exp_id, C_val in [("A_C1", 0.1), ("A_C2", 0.5), ("A_C3", 2.0), ("A_C4", 5.0), ("A_C5", 10.0)]:
    run_ablation(exp_id, f"Group A: C={C_val}", C=C_val, class_weight="balanced", chain_order=freq_desc_names)

best_c_id = max([k for k in all_results if k.startswith("A_C")], key=lambda k: all_results[k]['validation_metrics']['macro_f1'])
best_c_val = all_results[best_c_id]['hyperparameters']['C']
print(f"\n[BEST C: {best_c_val}]")

# ==========================================
# Group B: Chain Order
# ==========================================
orders = {
    "A_ORD1": ("Reverse frequency", rev_freq_desc_names, 42),
    "A_ORD2": ("Correlation-based", corr_names, 42),
    "A_ORD3": ("Alphabetical", alpha_names, 42),
    "A_ORD4": ("Rare-middle", rare_middle_names, 42),
}
for seed, name in [(42, "A_ORD5"), (0, "A_ORD6"), (123, "A_ORD7")]:
    np.random.seed(seed)
    rand_ord = list(freq_desc_names)
    np.random.shuffle(rand_ord)
    orders[name] = (f"Random seed {seed}", rand_ord, 42)

for exp_id, (desc, ord_names, seed) in orders.items():
    run_ablation(exp_id, f"Group B: {desc}", C=1.0, class_weight="balanced", chain_order=ord_names, chain_seed=seed)

# ==========================================
# Group C: Class Weight
# ==========================================
run_ablation("A_CW1", "Group C: class_weight=None, C=1.0", C=1.0, class_weight=None, chain_order=freq_desc_names)
if best_c_val != 1.0:
    run_ablation("A_CW2", f"Group C: class_weight=None, C={best_c_val}", C=best_c_val, class_weight=None, chain_order=freq_desc_names)


In [ ]:
!zip -r ablations_out.zip ablations_out/
from google.colab import files
files.download('ablations_out.zip')